# Monetization Model Hypothesis Test

## Research question

**H1:** `Free to Play (F2P)` games with `In-App Purchases` have a significantly lower Steam positive review rate than F2P games without `In-App Purchases`.

**H0:** The two groups have no significant difference in Steam positive review rate.

Available proxy variables:

- F2P: `games_clean.is_free == 1`, tag equals `Free to Play`, or genre equals `Free To Play`
- In-App Purchases: category equals `In-App Purchases`
- Review outcome: `positive / total`; games are retained only when `total >= 50` to reduce small-review-count noise


## 1. Load and split the data

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

DATA_DIR = Path("Cleaned_Data")
MIN_REVIEWS = 50
ALPHA = 0.05

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

games = pd.read_csv(DATA_DIR / "games_clean.csv")
reviews = pd.read_csv(DATA_DIR / "reviews_clean.csv")
categories = pd.read_csv(DATA_DIR / "categories_clean.csv")
tags = pd.read_csv(DATA_DIR / "tags_clean.csv")
genres = pd.read_csv(DATA_DIR / "genres_clean.csv")

In [2]:
# F2P indicators from three available sources.
free_by_game = set(games.loc[games["is_free"].astype(int).eq(1), "app_id"])
free_by_tag = set(tags.loc[tags["tag"].eq("Free to Play"), "app_id"])
free_by_genre = set(genres.loc[genres["genre"].eq("Free To Play"), "app_id"])
f2p_apps = free_by_game | free_by_tag | free_by_genre

# In-App Purchases is available as a cleaned Steam category.
iap_apps = set(categories.loc[categories["category"].eq("In-App Purchases"), "app_id"])

games_base = games.copy()
games_base["release_date"] = pd.to_datetime(games_base["release_date"], errors="coerce")
games_base["release_year"] = games_base["release_date"].dt.year
games_base["is_f2p_proxy"] = games_base["app_id"].isin(f2p_apps)
games_base["has_iap"] = games_base["app_id"].isin(iap_apps)

reviews_base = reviews.copy()
for col in ["positive", "negative", "total", "review_score"]:
    reviews_base[col] = pd.to_numeric(reviews_base[col], errors="coerce")
reviews_base["positive_rate"] = reviews_base["positive"] / reviews_base["total"]

df = games_base.merge(
    reviews_base[["app_id", "positive", "negative", "total", "review_score", "positive_rate"]],
    on="app_id",
    how="inner",
)

conditions = [
    df["is_f2p_proxy"] & df["has_iap"],
    df["is_f2p_proxy"] & ~df["has_iap"],
    ~df["is_f2p_proxy"] & df["has_iap"],
    ~df["is_f2p_proxy"] & ~df["has_iap"],
]
choices = ["F2P + IAP", "F2P no IAP", "Paid + IAP", "Paid no IAP"]
df["monetization_group"] = np.select(conditions, choices, default="Unknown")

analysis = df.loc[
    df["total"].ge(MIN_REVIEWS) & df["positive_rate"].notna(),
    [
        "app_id", "name", "release_date", "release_year", "price",
        "is_f2p_proxy", "has_iap", "monetization_group",
        "positive", "negative", "total", "review_score", "positive_rate",
    ],
].copy()

print("All games:", len(df))
print("F2P proxy apps:", len(f2p_apps))
print("In-App Purchases apps:", len(iap_apps))
print(f"Analysis sample with reviews >= {MIN_REVIEWS}:", len(analysis))
analysis["monetization_group"].value_counts()


All games: 140082
F2P proxy apps: 34413
In-App Purchases apps: 3264
Analysis sample with reviews >= 50: 32708


monetization_group
Paid no IAP    26048
F2P no IAP      5014
F2P + IAP       1354
Paid + IAP       292
Name: count, dtype: int64

## 2. Descriptive statistics

In [3]:
summary = (
    analysis.groupby("monetization_group")
    .agg(
        num_games=("app_id", "count"),
        mean_positive_rate=("positive_rate", "mean"),
        median_positive_rate=("positive_rate", "median"),
        std_positive_rate=("positive_rate", "std"),
        mean_review_score=("review_score", "mean"),
        median_review_count=("total", "median"),
    )
    .sort_values("mean_positive_rate", ascending=False)
)

summary.round(4)


,num_games,mean_positive_rate,median_positive_rate,std_positive_rate,mean_review_score,median_review_count
monetization_group,,,,,,
F2P no IAP,5014,0.8103,0.8492,0.1471,7.0642,168.0
Paid no IAP,26048,0.7861,0.8214,0.1561,6.8792,222.0
Paid + IAP,292,0.7333,0.7482,0.1494,6.3904,1801.5
F2P + IAP,1354,0.6857,0.6992,0.1478,5.9232,427.5


In [4]:
# A dependency-light bar view using pandas Styler. This avoids matplotlib/scipy requirements.
summary[["num_games", "mean_positive_rate", "mean_review_score"]].round(4).style.bar(
    subset=["mean_positive_rate"], color="#8fbcd4", vmin=0, vmax=1
)

,num_games,mean_positive_rate,mean_review_score
monetization_group,,,
F2P no IAP,5014,0.810300,7.064200
Paid no IAP,26048,0.786100,6.879200
Paid + IAP,292,0.733300,6.390400
F2P + IAP,1354,0.685700,5.923200


## 3. Primary hypothesis test

Primary comparison:

- Group A: `F2P + IAP`
- Group B: `F2P no IAP`

If Group A has a significantly lower positive review rate than Group B, the result supports the hypothesis that In-App Purchases in F2P games are associated with lower user evaluation.

In [5]:
a = analysis.loc[analysis["monetization_group"].eq("F2P + IAP"), "positive_rate"].dropna()
b = analysis.loc[analysis["monetization_group"].eq("F2P no IAP"), "positive_rate"].dropna()

# z = (mean_A - mean_B) / SE
se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
z  = (a.mean() - b.mean()) / se
p  = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / np.sqrt(2))))

print(f"F2P + IAP  mean positive rate: {a.mean():.4f}  (n={len(a)})")
print(f"F2P no IAP mean positive rate: {b.mean():.4f}  (n={len(b)})")
print(f"Difference (A - B)           : {a.mean() - b.mean():.4f}")
print(f"z = {z:.2f},  p = {p:.3e}")
print()
if p < ALPHA:
    print(f"p < {ALPHA}: reject H0 — F2P+IAP games have a significantly lower positive rate than F2P no IAP games, supporting H1.")
else:
    print(f"p >= {ALPHA}: fail to reject H0.")

F2P + IAP  mean positive rate: 0.6857  (n=1354)
F2P no IAP mean positive rate: 0.8103  (n=5014)
Difference (A - B)           : -0.1246
z = -27.56,  p = 0.000e+00

p < 0.05: reject H0 — F2P+IAP games have a significantly lower positive rate than F2P no IAP games, supporting H1.


**Result Interpretation**

The t-test yields z = −27.56 with p ≈ 0, far below the significance threshold of α = 0.05. We therefore reject the null hypothesis.

F2P games with In-App Purchases show a mean positive review rate of 68.6%, compared to 81.9% for F2P games without In-App Purchases — a gap of roughly 12.5 percentage points. With sample sizes of 1,354 and 5,014 respectively, this difference is too large and too consistent to be explained by random chance.

This result supports H1: the presence of In-App Purchases in free-to-play games is associated with meaningfully lower user satisfaction, as measured by Steam positive review rates.

Beside, we think Games with IAP may differ from those without in ways beyond monetization — for example, genre composition, development origin (mobile ports vs. PC-native), or publisher type. The genre-adjusted robustness check in Section 5 confirms the gap persists within individual genres, suggesting genre mix alone does not account for the full effect.

## 4. Additional comparisons

These comparisons are not the primary hypothesis test, but they help interpret the result:

- All IAP games vs all non-IAP games
- F2P no IAP vs Paid no IAP


In [6]:
comparisons = [
    ("F2P + IAP",  analysis["monetization_group"].eq("F2P + IAP"),  "F2P no IAP", analysis["monetization_group"].eq("F2P no IAP")),
    ("IAP",        analysis["has_iap"],                              "No IAP",     ~analysis["has_iap"]),
    ("F2P no IAP", analysis["monetization_group"].eq("F2P no IAP"), "Paid no IAP", analysis["monetization_group"].eq("Paid no IAP")),
]

rows = []
for left_name, left_mask, right_name, right_mask in comparisons:
    x = analysis.loc[left_mask,  "positive_rate"].dropna()
    y = analysis.loc[right_mask, "positive_rate"].dropna()
    se   = np.sqrt(x.var(ddof=1) / len(x) + y.var(ddof=1) / len(y))
    diff = x.mean() - y.mean()
    z    = diff / se
    p    = 2 * (1 - 0.5 * (1 + math.erf(abs(z) / np.sqrt(2))))
    rows.append({
        "comparison": f"{left_name} vs {right_name}",
        "n_a": len(x), "n_b": len(y),
        "mean_a": x.mean(), "mean_b": y.mean(),
        "diff (a-b)": diff,
        "ci95_low": diff - 1.96 * se, "ci95_high": diff + 1.96 * se,
        "z": z, "p_value": p,
    })

pd.DataFrame(rows).round(6)


,comparison,n_a,n_b,mean_a,mean_b,diff (a-b),ci95_low,ci95_high,z,p_value
0,F2P + IAP vs F2P no IAP,1354,5014,0.685690,0.810334,-0.124644,-0.133509,-0.115779,-27.557069,0.0
1,IAP vs No IAP,1646,31062,0.694132,0.790049,-0.095917,-0.103327,-0.088506,-25.367960,0.0
2,F2P no IAP vs Paid no IAP,5014,26048,0.810334,0.786144,0.024191,0.019700,0.028681,10.558111,0.0


## 5. Genre-adjusted robustness check

To reduce the potential effect of genre composition, this section performs a simple stratified check. It compares the average positive review rate within genres that contain both `F2P + IAP` and `F2P no IAP` games, then computes a weighted average using the smaller group size within each genre as the weight.=

In [7]:
game_genres = genres.merge(analysis[["app_id", "monetization_group", "positive_rate"]], on="app_id", how="inner")
target_genres = game_genres.loc[
    game_genres["monetization_group"].isin(["F2P + IAP", "F2P no IAP"])
]

genre_group = (
    target_genres.groupby(["genre", "monetization_group"])
    .agg(n=("app_id", "nunique"), mean_positive_rate=("positive_rate", "mean"))
    .reset_index()
)

genre_pivot = genre_group.pivot(index="genre", columns="monetization_group", values=["n", "mean_positive_rate"])
genre_pivot.columns = [f"{metric}_{group}" for metric, group in genre_pivot.columns]
genre_pivot = genre_pivot.dropna().copy()
genre_pivot = genre_pivot.loc[
    (genre_pivot["n_F2P + IAP"] >= 10) & (genre_pivot["n_F2P no IAP"] >= 10)
].copy()
genre_pivot["diff_iap_minus_no_iap"] = (
    genre_pivot["mean_positive_rate_F2P + IAP"] - genre_pivot["mean_positive_rate_F2P no IAP"]
)
genre_pivot["weight"] = genre_pivot[["n_F2P + IAP", "n_F2P no IAP"]].min(axis=1)

weighted_genre_diff = np.average(
    genre_pivot["diff_iap_minus_no_iap"], weights=genre_pivot["weight"]
) if len(genre_pivot) else np.nan

print("Weighted within-genre diff, F2P + IAP minus F2P no IAP:", round(weighted_genre_diff, 4))
genre_pivot.sort_values("diff_iap_minus_no_iap").round(4)


Weighted within-genre diff, F2P + IAP minus F2P no IAP: -0.1023


,n_F2P + IAP,n_F2P no IAP,mean_positive_rate_F2P + IAP,mean_positive_rate_F2P no IAP,diff_iap_minus_no_iap,weight
genre,,,,,,
Adventure,372.0,1915.0,0.6891,0.8177,-0.1286,372.0
RPG,487.0,831.0,0.6796,0.8063,-0.1267,487.0
Casual,561.0,1981.0,0.7021,0.8237,-0.1216,561.0
Free To Play,1156.0,3114.0,0.6884,0.8065,-0.1181,1156.0
Simulation,356.0,1022.0,0.6813,0.7967,-0.1154,356.0
Indie,560.0,3609.0,0.7147,0.8180,-0.1033,560.0
Action,568.0,1921.0,0.6857,0.7855,-0.0999,568.0
Sports,114.0,164.0,0.6842,0.7749,-0.0907,114.0
Strategy,531.0,758.0,0.7013,0.7815,-0.0802,531.0
